<a href="https://colab.research.google.com/github/thadduslee/Orbital-2026/blob/main/fine_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers peft datasets accelerate

In [2]:
!pip install --upgrade torchao

In [1]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType

# Load and preprocess Data
dataset_path = "/content/rebalanced_sentiment_reviews.csv"

# Read the CSV file
df = pd.read_csv(dataset_path)

# Drop any rows where 'message' or 'sentiment_score' are missing/null
df = df.dropna(subset=['message', 'sentiment_score'])

# REGRESSION FIX: Cast labels as floats.
# We no longer subtract 1, so the model will output directly on a 1.0 to 5.0 scale.
processed_data = {
    "text": df["message"].astype(str).tolist(),
    "label": df["sentiment_score"].astype(float).tolist()
}

dataset = Dataset.from_dict(processed_data)

# 80/20 train-test split
dataset = dataset.train_test_split(test_size=0.2, seed=42)

# Tokenizer and Base Model
model_id = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# REGRESSION FIX: Set num_labels=1 to output a single continuous value
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=1
)

# Low rank adaptation
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query_proj", "value_proj"], # Target attention layers
    modules_to_save=["classifier", "pooler"]     # CRITICAL FIX: Train and save the scoring head
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

# REGRESSION FIX: Standard Trainer automatically uses MSELoss for num_labels=1
training_args = TrainingArguments(
    output_dir="./module_scorer_results",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

# Save LoRA adapters
peft_model.save_pretrained("./deberta-lora-module-scorer")
tokenizer.save_pretrained("./deberta-lora-module-scorer")

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

trainable params: 886,273 || all params: 185,309,186 || trainable%: 0.4783


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,1.996112,1.759636
2,0.587287,0.580043
3,0.506375,0.614708
4,0.388618,0.382941
5,0.261206,0.359670


('./deberta-lora-module-scorer/tokenizer_config.json',
 './deberta-lora-module-scorer/tokenizer.json')

In [7]:
pip install transformers peft torch

In [9]:
import torch
from peft import PeftConfig, PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer

peft_model_id = "thaddus/deberta-lora-module-scorer"

config = PeftConfig.from_pretrained(peft_model_id)
base_model_name = config.base_model_name_or_path

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# CRITICAL FIX: Set num_labels=1 to explicitly tell Hugging Face
# this is a regression model predicting a single continuous value.
base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=1,
    ignore_mismatched_sizes=True # Prevents errors if base model had a different default
)

model = PeftModel.from_pretrained(base_model, peft_model_id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

comments = [
    "i hate this module, the professor was very disorganised and did not properly explain the material",
    "wonderful module, the professor was engaging and thoughtful and he went out of his way to ensure that we all understood the topic",
    "typical module, did not really like it, average workload and difficulty, finals was tricky but doable. just remain on the right side of the bell curve. only took the module because it was part of my core curriculum"
]

print(f"Running Regression Inference on {device}...\n" + "-"*40)

for comment in comments:
    inputs = tokenizer(
        comment,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        # Extract the single continuous score directly
        predicted_score = outputs.logits.squeeze().item()

    print(f"Comment: '{comment}'")

    # Optional: Clamp the score between 1 and 5 just in case it drifts
    clamped_score = max(1.0, min(5.0, predicted_score))
    print(f"Raw Score: {predicted_score:.4f}")
    print(f"Clamped Score (1-5): {clamped_score:.4f}\n")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

Running Regression Inference on cuda...
----------------------------------------
Comment: 'i hate this module, the professor was very disorganised and did not properly explain the material'
Raw Score: 1.5039
Clamped Score (1-5): 1.5039

Comment: 'wonderful module, the professor was engaging and thoughtful and he went out of his way to ensure that we all understood the topic'
Raw Score: 4.5234
Clamped Score (1-5): 4.5234

Comment: 'typical module, did not really like it, average workload and difficulty, finals was tricky but doable. just remain on the right side of the bell curve. only took the module because it was part of my core curriculum'
Raw Score: 2.3672
Clamped Score (1-5): 2.3672

